In [1]:
import requests
import pandas as pd
from sklearn.preprocessing import StandardScaler
# List of all MLB team IDs
team_ids = [
    108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121,
    133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 158
]
# Import data into tables
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'}
url = "https://baseballsavant.mlb.com/team/144?view=statcast&nav=hitting&season=2024"
response = requests.get(url, headers=headers)
tables = pd.read_html(response.text)
standard = tables[0]
statcast1 = tables[1]
statcast2 = tables[2]
# Clean the Names of the Columns
standard.columns = [' '.join(col).strip() if isinstance(col, tuple) else col for col in standard.columns]
statcast1.columns = [str(col) for col in statcast1.columns]
statcast2.columns = [str(col) for col in statcast2.columns]

unwanted = ['Unnamed: 1_level_0', 'Unnamed: 0_level_0', 'Statcast', 'Standard Stats','Unnamed: 26_level_1']

def clean_column(col):
    for u in unwanted:
        col = col.replace(u, '').strip()
    col = ' '.join(col.split())
    return col

for df in [standard, statcast1, statcast2]:
    new_columns = {}
    for col in df.columns:
        col_str = ' '.join(col).strip() if isinstance(col, tuple) else str(col)
        new_columns[col] = clean_column(col_str)
    df.rename(columns=new_columns, inplace=True)
# Organize Columns with repeats
statcast1 = statcast1.drop(columns=['Season','Pitches',])
statcast2 = statcast2.drop(columns=['Season','Barrel %'])
statcast3 = standard.drop(columns=[('Season'),('AB'),('H'),('2B'),('3B'),('HR'),('BB'), ('PA'),('SO'),('BA'),('OBP'),('SLG'),('WOBA'),('WOBACON')])
standard = standard.drop(columns=[('Season'),('PA'), ('SO'), ('BA'), ('OBP'), ('SLG'), ('WOBA'), ('WOBACON'),('Pitches'),('Batted Balls'),('Barrels'),('Barrel %'),('Hard Hit %'),('Exit Velocity'),('Launch Angle'),('XBA'),('XSLG'),('XWOBA'),('XWOBACON')])
# Create a standard stats and statcast stats tables
statcast3 = statcast3.rename(columns={'Unnamed: 0_level_0 Player': 'Player'})
statcast = pd.merge(statcast1, statcast2, on='Player', how='inner')
statcast = pd.merge(statcast, statcast3, on='Player', how='inner')
# Calculate "Expected Runs"
standard['TB'] = (
    standard['2B'] * 2 +
    standard['3B'] * 3 +
    standard['HR'] * 4 +
    (standard['H'] - standard['2B'] - standard['3B'] - standard['HR'])
)

standard['Runs'] = ((standard['H'] + standard['BB']) * standard['TB']) / (standard['AB'] + standard['BB'])
standard['Runs'] = standard['Runs'].round()
# Creating lists and diciontaries of wanted statcast numbers used in finding numbers for every player
hit_features = ['Whiff %','Batted Balls','Hard Hit %','Exit Velocity','XBA','Zone Contact %','Chase %', 'Zone Swing %', 'Straight %']
walk_features = ['Chase %', 'Batted Balls', 'Pitches', 'Whiff %', 'Zone Swing %', 'Zone Contact %']
bats_features = ['Pitches', 'Batted Balls']
bases_features = ['XSLG','Whiff %','Chase %','Barrel %', 'Hard Hit %', 'Exit Velocity', 'Launch Angle', 'Straight %', 'Batted Balls']

hit_betas = {'Whiff %': .6,'Batted Balls': .8,'Hard Hit %': 1,'Exit Velocity': .7,'XBA': .9,'Zone Contact %': .3,'Chase %': .5, 'Zone Swing %': .2, 'Straight %': .4}
walk_betas = {'Chase %': .5, 'Batted Balls': 1, 'Pitches': .8, 'Whiff %': .5, 'Zone Swing %': .1, 'Zone Contact %': .2}
bats_betas = {'Pitches': .3, 'Batted Balls': .7}
bases_betas = {'XSLG': .9,'Whiff %': .5,'Chase %': .5,'Barrel %': 1, 'Hard Hit %': .8, 'Exit Velocity': .7, 'Launch Angle': .3, 'Straight %': .4, 'Batted Balls': .3}
# Calculating inputs to data science experiments
all_features = set(hit_features + walk_features + bats_features + bases_features)
missing = [f for f in all_features if f not in statcast.columns]
if missing:
    print(f"Missing columns in statcast DataFrame: {missing}")
else:
    missing = [f for f in all_features if f not in statcast.columns]
if missing:
    print(f"Missing columns in statcast DataFrame: {missing}")
else:
    scaler = StandardScaler()
    statcast_scaled = statcast.copy()
    statcast_scaled[list(all_features)] = scaler.fit_transform(statcast[list(all_features)])
    
    def compute_statcast_score(df, features, betas):
        return sum(df[feat] * betas[feat] for feat in features)
    statcast['Hit_Score']   = compute_statcast_score(statcast_scaled, hit_features, hit_betas)
    statcast['Walk_Score']  = compute_statcast_score(statcast_scaled, walk_features, walk_betas)
    statcast['Bats_Score']  = compute_statcast_score(statcast_scaled, bats_features, bats_betas)
    statcast['Bases_Score'] = compute_statcast_score(statcast_scaled, bases_features, bases_betas)
    statcast['Hit_Score']   = statcast['Hit_Score'].round(3)
    statcast['Walk_Score']  = statcast['Walk_Score'].round(3)
    statcast['Bats_Score']  = statcast['Bats_Score'].round(3)
    statcast['Bases_Score'] = statcast['Bases_Score'].round(3)
# Importing the tables to csv files
standard.to_csv('Standard Batting.csv', index=False)
statcast.to_csv('Statcast.csv', index=False)

C:\Users\nncg7\AppData\Local\Temp\ipykernel_18520\746137158.py:13: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)
